In [1]:
from pathlib import Path
from datetime import datetime
import time

import pandas as pd
import numpy as np
import yfinance as yf
from tqdm.notebook import tqdm

In [2]:
PROJECT_ROOT = Path("..").resolve()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
UNIVERSE_DIR = RAW_DIR / "universe"
YF_DIR = RAW_DIR / "yfinance"
YF_META_DIR = YF_DIR / "metadata"

UNIVERSE_DIR.mkdir(parents=True, exist_ok=True)
YF_META_DIR.mkdir(parents=True, exist_ok=True)

UNIVERSE_PATH = UNIVERSE_DIR / "ind_nifty100list.csv"
VALIDATION_REPORT_PATH = YF_META_DIR / "ticker_validation_report.csv"

print("Project root:", PROJECT_ROOT)
print("Universe path:", UNIVERSE_PATH)

Project root: E:\Projects\marketguard-india
Universe path: E:\Projects\marketguard-india\data\raw\universe\ind_nifty100list.csv


In [3]:
universe_raw = pd.read_csv(UNIVERSE_PATH)

In [4]:
print("Shape:", universe_raw.shape)
display(universe_raw.head())
display(universe_raw.tail())
print(universe_raw.columns.tolist())

Shape: (100, 9)


,symbol,yf_ticker,company_name,industry,source,snapshot_date,active_for_v1,series,isin_code
0,ABB,ABB.NS,ABB India Ltd.,Capital Goods,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE117A01022
1,ADANIENSOL,ADANIENSOL.NS,Adani Energy Solutions Ltd.,Power,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE931S01010
2,ADANIENT,ADANIENT.NS,Adani Enterprises Ltd.,Metals & Mining,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE423A01024
3,ADANIGREEN,ADANIGREEN.NS,Adani Green Energy Ltd.,Power,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE364U01010
4,ADANIPORTS,ADANIPORTS.NS,Adani Ports and Special Economic Zone Ltd.,Services,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE742F01042


,symbol,yf_ticker,company_name,industry,source,snapshot_date,active_for_v1,series,isin_code
95,UNITDSPR,UNITDSPR.NS,United Spirits Ltd.,Fast Moving Consumer Goods,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE854D01024
96,VBL,VBL.NS,Varun Beverages Ltd.,Fast Moving Consumer Goods,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE200M01039
97,VEDL,VEDL.NS,Vedanta Ltd.,Metals & Mining,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE205A01025
98,WIPRO,WIPRO.NS,Wipro Ltd.,Information Technology,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE075A01022
99,ZYDUSLIFE,ZYDUSLIFE.NS,Zydus Lifesciences Ltd.,Healthcare,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE010B01027


['symbol', 'yf_ticker', 'company_name', 'industry', 'source', 'snapshot_date', 'active_for_v1', 'series', 'isin_code']


In [5]:
universe = universe_raw.copy()

universe.columns = (
    universe.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(".", "", regex=False)
)

print(universe.columns.tolist())
display(universe.head())

['symbol', 'yf_ticker', 'company_name', 'industry', 'source', 'snapshot_date', 'active_for_v1', 'series', 'isin_code']


,symbol,yf_ticker,company_name,industry,source,snapshot_date,active_for_v1,series,isin_code
0,ABB,ABB.NS,ABB India Ltd.,Capital Goods,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE117A01022
1,ADANIENSOL,ADANIENSOL.NS,Adani Energy Solutions Ltd.,Power,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE931S01010
2,ADANIENT,ADANIENT.NS,Adani Enterprises Ltd.,Metals & Mining,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE423A01024
3,ADANIGREEN,ADANIGREEN.NS,Adani Green Energy Ltd.,Power,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE364U01010
4,ADANIPORTS,ADANIPORTS.NS,Adani Ports and Special Economic Zone Ltd.,Services,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE742F01042


In [6]:
required_cols = ["symbol", "company_name", "industry"]

missing_cols = [col for col in required_cols if col not in universe.columns]
missing_cols

[]

In [7]:
universe = universe.copy()

universe["symbol"] = universe["symbol"].astype(str).str.strip().str.upper()
universe["company_name"] = universe["company_name"].astype(str).str.strip()
universe["industry"] = universe["industry"].astype(str).str.strip()

display(universe[["symbol", "company_name", "industry"]].head())

,symbol,company_name,industry
0,ABB,ABB India Ltd.,Capital Goods
1,ADANIENSOL,Adani Energy Solutions Ltd.,Power
2,ADANIENT,Adani Enterprises Ltd.,Metals & Mining
3,ADANIGREEN,Adani Green Energy Ltd.,Power
4,ADANIPORTS,Adani Ports and Special Economic Zone Ltd.,Services


### Add yfinance ticker for yahoo finance data dowload

In [8]:
universe["yf_ticker"] = universe["symbol"] + ".NS"

universe["source"] = "NSE/Nifty Indices current NIFTY 100 constituent file"
universe["snapshot_date"] = datetime.today().date().isoformat()
universe["active_for_v1"] = True

display(universe[["symbol", "yf_ticker", "company_name", "industry"]].head(20))

,symbol,yf_ticker,company_name,industry
0,ABB,ABB.NS,ABB India Ltd.,Capital Goods
1,ADANIENSOL,ADANIENSOL.NS,Adani Energy Solutions Ltd.,Power
2,ADANIENT,ADANIENT.NS,Adani Enterprises Ltd.,Metals & Mining
3,ADANIGREEN,ADANIGREEN.NS,Adani Green Energy Ltd.,Power
4,ADANIPORTS,ADANIPORTS.NS,Adani Ports and Special Economic Zone Ltd.,Services
5,ADANIPOWER,ADANIPOWER.NS,Adani Power Ltd.,Power
6,AMBUJACEM,AMBUJACEM.NS,Ambuja Cements Ltd.,Construction Materials
7,APOLLOHOSP,APOLLOHOSP.NS,Apollo Hospitals Enterprise Ltd.,Healthcare
8,ASIANPAINT,ASIANPAINT.NS,Asian Paints Ltd.,Consumer Durables
9,DMART,DMART.NS,Avenue Supermarts Ltd.,Consumer Services


In [9]:
summary = {
    "rows": len(universe),
    "unique_symbols": universe["symbol"].nunique(),
    "duplicate_symbols": universe["symbol"].duplicated().sum(),
    "missing_symbols": universe["symbol"].isna().sum(),
    "missing_company_names": universe["company_name"].isna().sum(),
    "missing_industries": universe["industry"].isna().sum(),
}

summary

{'rows': 100,
 'unique_symbols': 100,
 'duplicate_symbols': np.int64(0),
 'missing_symbols': np.int64(0),
 'missing_company_names': np.int64(0),
 'missing_industries': np.int64(0)}

In [10]:
duplicates = universe[universe["symbol"].duplicated(keep=False)].sort_values("symbol")
display(duplicates)

,symbol,yf_ticker,company_name,industry,source,snapshot_date,active_for_v1,series,isin_code


In [11]:
clean_cols = [
    "symbol",
    "yf_ticker",
    "company_name",
    "industry",
    "source",
    "snapshot_date",
    "active_for_v1",
]

# Keep optional useful columns if they exist
optional_cols = [col for col in ["series", "isin_code"] if col in universe.columns]

universe_clean = universe[clean_cols + optional_cols].copy()

universe_clean.to_csv(UNIVERSE_PATH, index=False)

display(universe_clean.head())
print("Saved:", UNIVERSE_PATH)

,symbol,yf_ticker,company_name,industry,source,snapshot_date,active_for_v1,series,isin_code
0,ABB,ABB.NS,ABB India Ltd.,Capital Goods,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE117A01022
1,ADANIENSOL,ADANIENSOL.NS,Adani Energy Solutions Ltd.,Power,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE931S01010
2,ADANIENT,ADANIENT.NS,Adani Enterprises Ltd.,Metals & Mining,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE423A01024
3,ADANIGREEN,ADANIGREEN.NS,Adani Green Energy Ltd.,Power,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE364U01010
4,ADANIPORTS,ADANIPORTS.NS,Adani Ports and Special Economic Zone Ltd.,Services,NSE/Nifty Indices current NIFTY 100 constituen...,2026-07-14,True,EQ,INE742F01042


Saved: E:\Projects\marketguard-india\data\raw\universe\ind_nifty100list.csv


In [15]:
VALIDATION_START = "2019-01-01"

def validate_yf_ticker(yf_ticker):
    try:
        df = yf.download(
            yf_ticker,
            start=VALIDATION_START,
            interval="1d",
            auto_adjust=False,
            progress=False,
            threads=False,
        )

        if df.empty:
            return {
                "status": "FAILED",
                "rows_returned": 0,
                "start_date": None,
                "end_date": None,
                "latest_close": np.nan,
                "has_ohlcv": False,
                "issue": "No data returned",
            }

        # yfinance may return MultiIndex columns. Flatten them.
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = [
                col[0] if isinstance(col, tuple) else col
                for col in df.columns
            ]

        df = df.reset_index()

        required_cols = ["Date", "Open", "High", "Low", "Close", "Volume"]
        missing_cols = [col for col in required_cols if col not in df.columns]

        if missing_cols:
            return {
                "status": "FAILED",
                "rows_returned": len(df),
                "start_date": df["Date"].min() if "Date" in df.columns else None,
                "end_date": df["Date"].max() if "Date" in df.columns else None,
                "latest_close": np.nan,
                "has_ohlcv": False,
                "issue": f"Missing columns: {missing_cols}",
            }

        price_cols = ["Open", "High", "Low", "Close"]
        bad_prices = (df[price_cols] <= 0).sum().sum()

        status = "OK"
        issue = ""

        if bad_prices > 0:
            status = "FLAGGED"
            issue = f"Non-positive price values found: {bad_prices}"

        latest_close = df["Close"].dropna().iloc[-1] if df["Close"].notna().any() else np.nan

        return {
            "status": status,
            "rows_returned": len(df),
            "start_date": df["Date"].min(),
            "end_date": df["Date"].max(),
            "latest_close": latest_close,
            "has_ohlcv": True,
            "issue": issue,
        }

    except Exception as e:
        return {
            "status": "ERROR",
            "rows_returned": 0,
            "start_date": None,
            "end_date": None,
            "latest_close": np.nan,
            "has_ohlcv": False,
            "issue": str(e),
        }

In [16]:
validate_yf_ticker("TCS.NS")

{'status': 'OK',
 'rows_returned': 1863,
 'start_date': Timestamp('2019-01-01 00:00:00'),
 'end_date': Timestamp('2026-07-14 00:00:00'),
 'latest_close': np.float64(2200.60009765625),
 'has_ohlcv': True,
 'issue': ''}

In [17]:
validation_rows = []

for _, row in tqdm(universe_clean.iterrows(), total=len(universe_clean)):
    result = validate_yf_ticker(row["yf_ticker"])

    validation_rows.append({
        "symbol": row["symbol"],
        "yf_ticker": row["yf_ticker"],
        "company_name": row["company_name"],
        "industry": row["industry"],
        **result,
    })

    time.sleep(0.15)

validation_report = pd.DataFrame(validation_rows)

display(validation_report.head())
validation_report["status"].value_counts()

  0%|          | 0/100 [00:00<?, ?it/s]

,symbol,yf_ticker,company_name,industry,status,rows_returned,start_date,end_date,latest_close,has_ohlcv,issue
0,ABB,ABB.NS,ABB India Ltd.,Capital Goods,OK,1863,2019-01-01,2026-07-14,6892.000000,True,
1,ADANIENSOL,ADANIENSOL.NS,Adani Energy Solutions Ltd.,Power,OK,1863,2019-01-01,2026-07-14,1690.300049,True,
2,ADANIENT,ADANIENT.NS,Adani Enterprises Ltd.,Metals & Mining,OK,1863,2019-01-01,2026-07-14,3188.899902,True,
3,ADANIGREEN,ADANIGREEN.NS,Adani Green Energy Ltd.,Power,OK,1863,2019-01-01,2026-07-14,1604.500000,True,
4,ADANIPORTS,ADANIPORTS.NS,Adani Ports and Special Economic Zone Ltd.,Services,OK,1863,2019-01-01,2026-07-14,1821.900024,True,


status
OK    100
Name: count, dtype: int64

In [18]:
failed_or_flagged = validation_report[
    validation_report["status"].isin(["FAILED", "ERROR", "FLAGGED"])
].copy()

display(failed_or_flagged)

,symbol,yf_ticker,company_name,industry,status,rows_returned,start_date,end_date,latest_close,has_ohlcv,issue


In [19]:
validation_report.to_csv(VALIDATION_REPORT_PATH, index=False)

print("Saved validation report:", VALIDATION_REPORT_PATH)

display(
    validation_report
    .groupby("status")
    .agg(
        tickers=("yf_ticker", "count"),
        min_rows=("rows_returned", "min"),
        max_rows=("rows_returned", "max"),
    )
)

Saved validation report: E:\Projects\marketguard-india\data\raw\yfinance\metadata\ticker_validation_report.csv


,tickers,min_rows,max_rows
status,,,
OK,100,168,1863
